# 📊 Interactive Stock & Gold Analysis System
### UQU-DS-2025-M03 | Umm Al-Qura University — Data Science Department
**Authors:** Marwan Al Otaibi · Yazeed Al Sayari · Abdulmohsen Asiri · Nawaf Aladwani

---
This notebook implements:
- **Task 1 — Sentiment Analysis:** VADER vs FinBERT vs RoBERTa evaluated against ground-truth labels
- **Task 2 — Price Prediction:** XGBoost vs Linear Regression vs Random Forest on AAPL & Gold

**Before running:** Upload `apple_news_clean.xlsx` and `gold_news_clean.xlsx` to the Colab session files panel (left sidebar → Files icon).


## ⚙️ Section 1 — Install & Import

In [ ]:
# Install required packages
# transformers + torch for RoBERTa
# nltk for text preprocessing (Logistic Regression)
# yfinance for price data
# xgboost for gradient boosting
!pip install transformers torch vaderSentiment yfinance xgboost plotly -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.7 MB/s eta 0:00:00


In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ── Torch / GPU setup ────────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — running on CPU (FinBERT/RoBERTa will be slower)')


Device: cuda
GPU: Tesla T4


## 📰 Section 2 — Load News Data

Upload `apple_news_clean.xlsx` and `gold_news_clean.xlsx` using the **Files** panel on the left sidebar, then run this cell.


In [ ]:
# ── Arabic column names → English ────────────────────────────────────────────
COL_MAP = {
    'التاريخ':           'date',
    'العنوان':           'headline',
    'الوصف':             'description',
    'المصدر':            'source',
    'الكلمة':            'keyword',
    'Sentiment الكامل':  'sentiment_full',
    'تصنيف Sentiment':   'sentiment_label',
    'الذاتية':           'subjectivity',
    'Sentiment العنوان': 'headline_sentiment',
    'تصنيف العنوان':     'headline_label',
    'يوم الأسبوع':       'day_of_week',
    'اسم اليوم':         'day_name',
    'الشهر':             'month',
    'الربع':             'quarter',
}

def load_news(path):
    df = pd.read_excel(path)
    df = df.rename(columns=COL_MAP)
    df['date'] = pd.to_datetime(df['date'])
    keep = ['date', 'headline', 'description', 'sentiment_label']
    df = df[[c for c in keep if c in df.columns]].copy()

    # IMPORTANT: score headline text ONLY — not headline + description.
    #
    # Root cause of the VADER 1.00 accuracy bug:
    #   The ground-truth labels were generated by running VADER on the
    #   concatenated headline + description text (verified: the stored
    #   'Sentiment الكامل' scores match VADER compound on that combined
    #   text exactly — difference is 0.000 on every row).
    #   Feeding the same text back into VADER during evaluation is
    #   circular — it scores its own output and trivially gets 1.00.
    #
    # Fix — use headline only:
    #   1. Fair comparison: none of the three models sees the text
    #      that generated the ground truth.
    #   2. Appropriate input size: FinBERT and RoBERTa are designed
    #      for short financial text (512-token limit); headlines fit
    #      naturally and descriptions often exceed it.
    #   3. Realistic benchmark: VADER achieves ~61% on headlines —
    #      a genuine signal to compare against FinBERT and RoBERTa.
    df['text'] = df['headline'].fillna('').str.strip()
    df['sentiment_label'] = df['sentiment_label'].str.strip().str.capitalize()
    df = df.dropna(subset=['sentiment_label', 'text'])
    df = df[df['text'] != '']
    return df

apple_news = load_news('/content/apple_news_clean.xlsx')
gold_news  = load_news('/content/gold_news_clean.xlsx')

print(f'Apple news: {len(apple_news):,} articles  '
      f'({apple_news.date.min().date()} -> {apple_news.date.max().date()})')
print(f'Gold  news: {len(gold_news):,} articles  '
      f'({gold_news.date.min().date()} -> {gold_news.date.max().date()})')
print()
print('Apple label distribution:')
print(apple_news.sentiment_label.value_counts(normalize=True).round(3))
print()
print('Gold label distribution:')
print(gold_news.sentiment_label.value_counts(normalize=True).round(3))
print()
print('NOTE: Scoring uses headline text only (see comment above for reasoning).')


Apple news: 2,185 articles  (2025-01-02 -> 2026-05-10)
Gold  news: 2,854 articles  (2025-12-26 -> 2026-05-12)

Apple label distribution:
sentiment_label
Positive    0.664
Negative    0.187
Neutral     0.149
Name: proportion, dtype: float64

Gold label distribution:
sentiment_label
Negative    0.437
Positive    0.433
Neutral     0.131
Name: proportion, dtype: float64

NOTE: Scoring uses headline text only (see comment above for reasoning).


In [ ]:
# ============================================================
# SECTION 2 — PART 2: تحميل ودمج البيانات القديمة (2020–2024)
# ============================================================

def load_old_news(path, asset_name):
    """
    تحميل ملفات CSV القديمة وتوحيد شكلها
    مع ملفات Excel الجديدة
    """
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])

    # توحيد أسماء الأعمدة
    df = df.rename(columns={'title': 'headline',
                             'cleaned_title': 'cleaned_headline'})

    # استخدم cleaned_headline إذا موجود وإلا headline
    df['text'] = df['cleaned_headline'].fillna(df['headline']).fillna('').str.strip()

    # إضافة عمود sentiment_label فارغ — سيتم تسكورته بالنماذج
    df['sentiment_label'] = None

    df = df[['date', 'headline', 'text', 'sentiment_label']].copy()
    df = df[df['text'] != ''].dropna(subset=['text'])

    return df

# تحميل ودمج ملفات Apple
apple_old_part1 = load_old_news('/content/apple_cleaned_dataset.csv', 'AAPL')
apple_old_part2 = load_old_news('/content/apple_cleaned_dataset1.csv', 'AAPL')
apple_old = pd.concat([apple_old_part1, apple_old_part2], ignore_index=True)

# تحميل ودمج ملفات Gold
gold_old_part1 = load_old_news('/content/gold_cleaned_dataset.csv', 'Gold')
gold_old_part2 = load_old_news('/content/gold_cleaned_dataset1.csv', 'Gold')
gold_old = pd.concat([gold_old_part1, gold_old_part2], ignore_index=True)

print(f'AAPL total old news: {len(apple_old):,} articles ({apple_old.date.min().date()} → {apple_old.date.max().date()})')
print(f'Gold total old news: {len(gold_old):,} articles ({gold_old.date.min().date()} → {gold_old.date.max().date()})')

AAPL total old news: 16,727 articles (2020-01-03 → 2025-04-27)
Gold total old news: 7,021 articles (2020-01-07 → 2025-05-01)


In [ ]:
# ============================================================
# SECTION 2 — PART 3: دمج البيانات القديمة مع الجديدة
# ============================================================

def merge_news(new_df, old_df, asset_name):
    """
    دمج ملفي الأخبار:
    - الجديد (Excel): عنده ground truth labels → يُستخدم للتقييم والتدريب
    - القديم (CSV):   بدون labels → يُستخدم للـ sentiment scoring فقط
    """
    # الجديد: له labels
    new_df['has_label'] = True

    # القديم: بدون labels
    old_df['has_label'] = False

    combined = pd.concat([old_df, new_df], ignore_index=True)
    combined = combined.sort_values('date').reset_index(drop=True)

    print(f'\n{asset_name} combined:')
    print(f'  Total articles  : {len(combined):,}')
    print(f'  Date range      : {combined.date.min().date()} → {combined.date.max().date()}')
    print(f'  With labels     : {combined.has_label.sum():,}')
    print(f'  Without labels  : {(~combined.has_label).sum():,}')
    return combined

apple_news_full = merge_news(apple_news, apple_old, 'AAPL')
gold_news_full  = merge_news(gold_news,  gold_old,  'Gold')


AAPL combined:
  Total articles  : 18,912
  Date range      : 2020-01-03 → 2026-05-10
  With labels     : 2,185
  Without labels  : 16,727

Gold combined:
  Total articles  : 9,875
  Date range      : 2020-01-07 → 2026-05-12
  With labels     : 2,854
  Without labels  : 7,021


In [ ]:
# ============================================================
# SECTION 2 — PART 4: تحديث score_dataset لاستخدام البيانات الكاملة
# ============================================================

# للتقييم (Accuracy/F1) نستخدم فقط المقالات التي عندها labels
apple_news_eval = apple_news_full[apple_news_full['has_label']].copy()
gold_news_eval  = gold_news_full[gold_news_full['has_label']].copy()

# للـ sentiment scoring والـ aggregation نستخدم كل البيانات
apple_news = apple_news_full.copy()
gold_news  = gold_news_full.copy()

print(f'AAPL eval set  : {len(apple_news_eval):,} articles (with ground truth)')
print(f'AAPL full set  : {len(apple_news):,} articles (for scoring)')
print()
print(f'Gold eval set  : {len(gold_news_eval):,} articles (with ground truth)')
print(f'Gold full set  : {len(gold_news):,} articles (for scoring)')

AAPL eval set  : 2,185 articles (with ground truth)
AAPL full set  : 18,912 articles (for scoring)

Gold eval set  : 2,854 articles (with ground truth)
Gold full set  : 9,875 articles (for scoring)


## Section 3 — Sentiment Analysis: VADER vs Logistic Regression vs RoBERTa


In [ ]:
# ============================================================
# SECTION 3 — PART 1: VADER
# ============================================================

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

def run_vader(texts):
    analyzer = SentimentIntensityAnalyzer()
    labels, scores = [], []
    for t in texts:
        s = analyzer.polarity_scores(str(t))
        scores.append(s['compound'])
        if s['compound'] >= 0.05:
            labels.append('Positive')
        elif s['compound'] <= -0.05:
            labels.append('Negative')
        else:
            labels.append('Neutral')
    return labels, scores

print('VADER ready.')

VADER ready.


In [ ]:
# ============================================================
# SECTION 3 — PART 2: Logistic Regression Sentiment
# ============================================================

from sklearn.linear_model            import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline                import Pipeline

def train_logistic_sentiment(train_texts, train_labels):
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            strip_accents='unicode',
            analyzer='word',
        )),
        ('clf', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            C=1.0,
            random_state=42,
        ))
    ])
    pipeline.fit(train_texts, train_labels)
    return pipeline

def run_logistic(texts, labels, asset_name):
    # تدريب على 80% من البيانات التي عندها labels فقط
    labeled_mask  = labels.notna()
    labeled_texts = texts[labeled_mask].tolist()
    labeled_lbls  = labels[labeled_mask].tolist()

    split       = int(len(labeled_texts) * 0.8)
    train_texts = labeled_texts[:split]
    train_lbls  = labeled_lbls[:split]

    print(f'[{asset_name}] Training Logistic Regression '
          f'on {len(train_texts):,} labeled articles...')

    model  = train_logistic_sentiment(train_texts, train_lbls)
    preds  = model.predict(texts.tolist())
    probas = model.predict_proba(texts.tolist())

    classes = list(model.classes_)
    pos_idx = classes.index('Positive') if 'Positive' in classes else 0
    neg_idx = classes.index('Negative') if 'Negative' in classes else 1
    scores  = probas[:, pos_idx] - probas[:, neg_idx]

    print(f'[{asset_name}] Logistic Regression done.')
    return list(preds), list(scores), model

print('Logistic Regression ready.')

Logistic Regression ready.


In [ ]:
# ============================================================
# SECTION 3 — PART 2B: Text Preprocessing for Logistic Regression
# ============================================================

import re
import string
import nltk
from nltk.tokenize        import word_tokenize
from nltk.corpus          import stopwords
from nltk.stem            import WordNetLemmatizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline     import Pipeline

# تحميل المكتبات المطلوبة
nltk.download('punkt',        quiet=True)
nltk.download('punkt_tab',    quiet=True)
nltk.download('stopwords',    quiet=True)
nltk.download('wordnet',      quiet=True)
nltk.download('omw-1.4',      quiet=True)

lemmatizer    = WordNetLemmatizer()
stop_words    = set(stopwords.words('english'))

# كلمات مالية مهمة لا نحذفها من الـ stopwords
FINANCIAL_KEEP = {
    'up', 'down', 'high', 'low', 'not', 'no', 'but',
    'above', 'below', 'against', 'over', 'under',
    'bull', 'bear', 'rise', 'fall', 'gain', 'loss',
    'strong', 'weak', 'positive', 'negative'
}
stop_words = stop_words - FINANCIAL_KEEP

print('NLTK libraries downloaded.')

NLTK libraries downloaded.


In [ ]:
# ============================================================
# SECTION 3 — PART 2C: Preprocessing Pipeline
# ============================================================

def preprocess_text(text):
    """
    خطوات المعالجة المسبقة الكاملة:
    1. Lowercasing       — تحويل للأحرف الصغيرة
    2. Remove URLs       — إزالة الروابط
    3. Remove mentions   — إزالة @mentions و #hashtags
    4. Remove numbers    — إزالة الأرقام
    5. Remove punctuation— إزالة علامات الترقيم
    6. Tokenization      — تقسيم النص لكلمات
    7. Remove stopwords  — إزالة الكلمات الشائعة
    8. Lemmatization     — إرجاع الكلمة لجذرها
    """
    if not isinstance(text, str) or text.strip() == '':
        return ''

    # 1. Lowercasing
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # 3. Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)

    # 4. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 5. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 6. Tokenization
    tokens = word_tokenize(text)

    # 7. Remove stopwords + short tokens
    tokens = [t for t in tokens
              if t not in stop_words and len(t) > 2]

    # 8. Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)


def preprocess_batch(texts, asset_name=''):
    """تطبيق المعالجة على مجموعة من النصوص مع progress"""
    processed = []
    total     = len(texts)
    for i, text in enumerate(texts):
        processed.append(preprocess_text(text))
        if (i + 1) % 500 == 0:
            print(f'  [{asset_name}] Preprocessed {i+1}/{total}', end='\r')
    print(f'  [{asset_name}] Preprocessing done — {total:,} texts')
    return processed


# ── مثال توضيحي ────────────────────────────────────────────
sample = "Apple's stock price SOARED 15% after Q3 earnings! https://t.co/xyz #AAPL @TimCook"
print('Original  :', sample)
print('Processed :', preprocess_text(sample))

Original  : Apple's stock price SOARED 15% after Q3 earnings! https://t.co/xyz #AAPL @TimCook
Processed : apple stock price soared earnings


In [ ]:
# ============================================================
# SECTION 3 — PART 2C: Preprocessing Pipeline
# ============================================================

def preprocess_text(text):
    """
    خطوات المعالجة المسبقة الكاملة:
    1. Lowercasing       — تحويل للأحرف الصغيرة
    2. Remove URLs       — إزالة الروابط
    3. Remove mentions   — إزالة @mentions و #hashtags
    4. Remove numbers    — إزالة الأرقام
    5. Remove punctuation— إزالة علامات الترقيم
    6. Tokenization      — تقسيم النص لكلمات
    7. Remove stopwords  — إزالة الكلمات الشائعة
    8. Lemmatization     — إرجاع الكلمة لجذرها
    """
    if not isinstance(text, str) or text.strip() == '':
        return ''

    # 1. Lowercasing
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # 3. Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)

    # 4. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 5. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 6. Tokenization
    tokens = word_tokenize(text)

    # 7. Remove stopwords + short tokens
    tokens = [t for t in tokens
              if t not in stop_words and len(t) > 2]

    # 8. Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)


def preprocess_batch(texts, asset_name=''):
    """تطبيق المعالجة على مجموعة من النصوص مع progress"""
    processed = []
    total     = len(texts)
    for i, text in enumerate(texts):
        processed.append(preprocess_text(text))
        if (i + 1) % 500 == 0:
            print(f'  [{asset_name}] Preprocessed {i+1}/{total}', end='\r')
    print(f'  [{asset_name}] Preprocessing done — {total:,} texts')
    return processed


# ── مثال توضيحي ────────────────────────────────────────────
sample = "Apple's stock price SOARED 15% after Q3 earnings! https://t.co/xyz #AAPL @TimCook"
print('Original  :', sample)
print('Processed :', preprocess_text(sample))

Original  : Apple's stock price SOARED 15% after Q3 earnings! https://t.co/xyz #AAPL @TimCook
Processed : apple stock price soared earnings


In [ ]:
# ============================================================
# SECTION 3 — PART 2D: Apply Preprocessing to All News Data
# ============================================================

print('Preprocessing AAPL news...')
apple_news['text_processed'] = preprocess_batch(
    apple_news['text'].tolist(), 'AAPL'
)

print('\nPreprocessing Gold news...')
gold_news['text_processed'] = preprocess_batch(
    gold_news['text'].tolist(), 'Gold'
)

# إحصائيات بعد المعالجة
def show_preprocessing_stats(df, asset_name):
    original_len  = df['text'].str.split().str.len().mean()
    processed_len = df['text_processed'].str.split().str.len().mean()
    print(f'\n{asset_name} Preprocessing Stats:')
    print(f'  Avg words before : {original_len:.1f}')
    print(f'  Avg words after  : {processed_len:.1f}')
    print(f'  Reduction        : {(1 - processed_len/original_len)*100:.1f}%')
    print(f'\n  Sample before: {df["text"].iloc[0][:80]}...')
    print(f'  Sample after : {df["text_processed"].iloc[0][:80]}...')

show_preprocessing_stats(apple_news, 'AAPL')
show_preprocessing_stats(gold_news,  'Gold')

Preprocessing AAPL news...
  [AAPL] Preprocessing done — 18,912 texts

Preprocessing Gold news...
  [Gold] Preprocessing done — 9,875 texts

AAPL Preprocessing Stats:
  Avg words before : 12.5
  Avg words after  : 8.2
  Reduction        : 33.9%

  Sample before: apple ceo tim cook earned over $11 million in 2019, not counting stock awards...
  Sample after : apple ceo tim cook earned over million not counting stock award...

Gold Preprocessing Stats:
  Avg words before : 12.0
  Avg words after  : 8.1
  Reduction        : 32.8%

  Sample before: rickards here s where gold will be in 2026...
  Sample after : rickards gold...


In [ ]:
# ============================================================
# SECTION 3 — PART 2E: Updated Logistic Regression (with preprocessing)
# ============================================================

def train_logistic_sentiment(train_texts, train_labels):
    """
    Pipeline كامل:
    TF-IDF Vectorizer → Logistic Regression
    يستخدم النصوص المعالجة مسبقاً
    """
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features   = 8000,     # أكثر features بعد تنظيف النص
            ngram_range    = (1, 3),   # unigrams + bigrams + trigrams
            min_df         = 2,
            max_df         = 0.95,     # إزالة الكلمات الشائعة جداً
            sublinear_tf   = True,     # log(TF) بدل TF — يحسن الأداء
            strip_accents  = 'unicode',
            analyzer       = 'word',
        )),
        ('clf', LogisticRegression(
            class_weight = 'balanced',
            max_iter     = 1000,
            C            = 1.0,
            solver       = 'lbfgs',
            random_state = 42,
        ))
    ])
    pipeline.fit(train_texts, train_labels)
    return pipeline


def run_logistic(df, asset_name):
    """
    تدريب واختبار Logistic Regression
    يستخدم text_processed بدل text
    """
    # استخدم النصوص المعالجة
    texts  = df['text_processed']
    labels = df['sentiment_label']

    # تدريب على البيانات التي عندها labels فقط
    labeled_mask  = labels.notna()
    labeled_texts = texts[labeled_mask].tolist()
    labeled_lbls  = labels[labeled_mask].tolist()

    split       = int(len(labeled_texts) * 0.8)
    train_texts = labeled_texts[:split]
    train_lbls  = labeled_lbls[:split]

    print(f'[{asset_name}] Training Logistic Regression '
          f'on {len(train_texts):,} preprocessed articles...')

    model  = train_logistic_sentiment(train_texts, train_lbls)

    # تنبؤ على كل البيانات
    all_texts = texts.tolist()
    preds     = model.predict(all_texts)
    probas    = model.predict_proba(all_texts)

    classes = list(model.classes_)
    pos_idx = classes.index('Positive') if 'Positive' in classes else 0
    neg_idx = classes.index('Negative') if 'Negative' in classes else 1
    scores  = probas[:, pos_idx] - probas[:, neg_idx]

    print(f'[{asset_name}] Logistic Regression done.')
    return list(preds), list(scores), model

In [ ]:
# ============================================================
# SECTION 3 — PART 2F: Top TF-IDF Words Visualization
# ============================================================

def plot_top_tfidf_words(model, asset_name, top_n=15):
    """
    عرض أهم الكلمات لكل فئة sentiment
    """
    tfidf    = model.named_steps['tfidf']
    clf      = model.named_steps['clf']
    features = tfidf.get_feature_names_out()
    classes  = clf.classes_

    colors = {'Positive': '#3fb950', 'Negative': '#f85149', 'Neutral': '#d29922'}

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=list(classes))

    for col_idx, cls in enumerate(classes, 1):
        cls_idx    = list(classes).index(cls)
        coefs      = clf.coef_[cls_idx]
        top_idx    = np.argsort(coefs)[-top_n:]
        top_words  = [features[i] for i in top_idx]
        top_scores = [coefs[i]    for i in top_idx]

        fig.add_trace(go.Bar(
            x=top_scores,
            y=top_words,
            orientation='h',
            marker_color=colors.get(cls, '#58a6ff'),
            showlegend=False,
        ), row=1, col=col_idx)

    fig.update_layout(
        title=f'{asset_name} — Top {top_n} TF-IDF Words per Sentiment Class',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=500,
    )
    fig.show()


# سيتم استدعاؤها بعد التدريب في score_dataset
print('Preprocessing pipeline ready.')

Preprocessing pipeline ready.


In [ ]:
# ============================================================
# SECTION 3 — PART 3: RoBERTa
# ============================================================

import torch
from transformers import pipeline as hf_pipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

CACHE_DIR = '/content/sentiment_cache'
import os
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(asset, model):
    return os.path.join(CACHE_DIR, f'{asset}_{model}.csv')

def load_cache(asset, model):
    p = cache_path(asset, model)
    if os.path.exists(p):
        return pd.read_csv(p)
    return None

def save_cache(df, asset, model):
    df.to_csv(cache_path(asset, model), index=False)

def load_roberta():
    return hf_pipeline(
        'text-classification',
        model='cardiffnlp/twitter-roberta-base-sentiment-latest',
        device=0 if DEVICE == 'cuda' else -1,
        truncation=True,
        max_length=512,
    )

def run_transformer(pipe, texts, batch_size=32):
    labels, scores = [], []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch   = list(texts[i:i+batch_size])
        results = pipe(batch, truncation=True, max_length=512)
        for r in results:
            raw = r['label'].strip().capitalize()
            if raw in ('Positive', 'Pos'):
                labels.append('Positive')
            elif raw in ('Negative', 'Neg'):
                labels.append('Negative')
            else:
                labels.append('Neutral')
            scores.append(r['score'] if raw not in ('Negative','Neg')
                          else -r['score'])
        if (i // batch_size) % 5 == 0:
            print(f'  {min(i+batch_size, total)}/{total}', end='\r')
    print()
    return labels, scores

print('RoBERTa functions ready.')

Device: cuda
RoBERTa functions ready.


In [ ]:
# ============================================================
# SECTION 3 — PART 4 (UPDATED): Score Dataset with Preprocessing
# ============================================================

def score_dataset(df, asset_name):
    results = df[['date', 'sentiment_label']].copy()

    # ── VADER — يستخدم النص الأصلي ───────────────────────────────────────────
    print(f'\n[{asset_name}] Running VADER...')
    v_labels, v_scores = run_vader(df['text'].tolist())
    results['vader_label'] = v_labels
    results['vader_score'] = v_scores
    print(f'[{asset_name}] VADER done.')

    # ── Logistic Regression — يستخدم النص المعالج ────────────────────────────
    # تأكد أن text_processed موجود
    if 'text_processed' not in df.columns:
        print(f'[{asset_name}] Preprocessing text first...')
        df = df.copy()
        df['text_processed'] = preprocess_batch(df['text'].tolist(), asset_name)

    print(f'\n[{asset_name}] Running Logistic Regression (preprocessed text)...')
    lr_labels, lr_scores, lr_model = run_logistic(df, asset_name)
    results['finbert_label'] = lr_labels
    results['finbert_score'] = lr_scores

    # عرض أهم الكلمات
    plot_top_tfidf_words(lr_model, asset_name)

    # ── RoBERTa ───────────────────────────────────────────────────────────────
    cached = load_cache(asset_name, 'roberta')
    if cached is not None:
        print(f'\n[{asset_name}] RoBERTa — loaded from cache '
              f'({len(cached):,} rows)')
        results['roberta_label'] = cached['roberta_label'].values
        results['roberta_score'] = cached['roberta_score'].values
    else:
        print(f'\n[{asset_name}] Running RoBERTa on {len(df):,} articles...')
        pipe = load_roberta()
        rb_labels, rb_scores = run_transformer(pipe, df['text'].tolist())
        results['roberta_label'] = rb_labels
        results['roberta_score'] = rb_scores
        save_cache(
            results[['roberta_label', 'roberta_score']],
            asset_name, 'roberta'
        )
        del pipe
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    return results


# ── احذف الكاش القديم وأعد التشغيل ──────────────────────────────────────────
import shutil
shutil.rmtree('/content/sentiment_cache', ignore_errors=True)
os.makedirs('/content/sentiment_cache', exist_ok=True)
print('Cache cleared — ready to score with preprocessed text.')

apple_scored = score_dataset(apple_news, 'apple')
gold_scored  = score_dataset(gold_news,  'gold')

print('\nScoring complete.')
print(f'Apple scored: {len(apple_scored):,} articles')
print(f'Gold  scored: {len(gold_scored):,} articles')

Cache cleared — ready to score with preprocessed text.

[apple] Running VADER...
[apple] VADER done.

[apple] Running Logistic Regression (preprocessed text)...
[apple] Training Logistic Regression on 1,748 preprocessed articles...
[apple] Logistic Regression done.



[apple] Running RoBERTa on 18,912 articles...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

  18912/18912

[gold] Running VADER...
[gold] VADER done.

[gold] Running Logistic Regression (preprocessed text)...
[gold] Training Logistic Regression on 2,283 preprocessed articles...
[gold] Logistic Regression done.



[gold] Running RoBERTa on 9,875 articles...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




Scoring complete.
Apple scored: 18,912 articles
Gold  scored: 9,875 articles


In [ ]:
# ============================================================
# SECTION 3 — PART 5 (UPDATED): run_logistic مع text_processed
# ============================================================

def run_logistic(df, asset_name):
    """
    يستخدم text_processed بدل text
    """
    # استخدم النصوص المعالجة
    texts  = df['text_processed']
    labels = df['sentiment_label']

    # تدريب على البيانات التي عندها labels فقط
    labeled_mask  = labels.notna()
    labeled_texts = texts[labeled_mask].tolist()
    labeled_lbls  = labels[labeled_mask].tolist()

    split       = int(len(labeled_texts) * 0.8)
    train_texts = labeled_texts[:split]
    train_lbls  = labeled_lbls[:split]

    print(f'[{asset_name}] Training on {len(train_texts):,} '
          f'preprocessed articles...')

    model  = train_logistic_sentiment(train_texts, train_lbls)

    # تنبؤ على كل البيانات
    all_texts = texts.tolist()
    preds     = model.predict(all_texts)
    probas    = model.predict_proba(all_texts)

    classes = list(model.classes_)
    pos_idx = classes.index('Positive') if 'Positive' in classes else 0
    neg_idx = classes.index('Negative') if 'Negative' in classes else 1
    scores  = probas[:, pos_idx] - probas[:, neg_idx]

    print(f'[{asset_name}] Logistic Regression done.')
    return list(preds), list(scores), model

In [ ]:
# ============================================================
# SECTION 3 — PART 6: Evaluate (على المقالات التي عندها labels فقط)
# ============================================================

from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, classification_report)

def evaluate_sentiment(scored_df, asset_name):
    """
    التقييم على المقالات الجديدة فقط (التي عندها ground truth labels)
    """
    # فلتر المقالات التي عندها labels
    eval_df = scored_df[scored_df['sentiment_label'].notna()].copy()
    print(f'\n[{asset_name}] Evaluating on {len(eval_df):,} labeled articles...')

    y_true = eval_df['sentiment_label']
    models = {
        'VADER'               : eval_df['vader_label'],
        'Logistic Regression' : eval_df['finbert_label'],
        'RoBERTa'             : eval_df['roberta_label'],
    }

    rows = []
    for name, y_pred in models.items():
        rows.append({
            'Model'    : name,
            'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
            'Precision': round(precision_score(y_true, y_pred,
                               average='weighted', zero_division=0), 4),
            'Recall'   : round(recall_score(y_true, y_pred,
                               average='weighted', zero_division=0), 4),
            'F1-Score' : round(f1_score(y_true, y_pred,
                               average='weighted', zero_division=0), 4),
        })
        print(f'\n--- {asset_name} | {name} ---')
        print(classification_report(y_true, y_pred, zero_division=0))

    tbl = pd.DataFrame(rows)
    print(f'\n=== {asset_name} — Sentiment Model Comparison ===')
    print(tbl.to_string(index=False))
    return tbl

apple_sent_metrics = evaluate_sentiment(apple_scored, 'AAPL')
gold_sent_metrics  = evaluate_sentiment(gold_scored,  'Gold')


[AAPL] Evaluating on 2,185 labeled articles...

--- AAPL | VADER ---
              precision    recall  f1-score   support

    Negative       0.62      0.61      0.61       408
     Neutral       0.32      0.92      0.47       326
    Positive       0.94      0.54      0.69      1451

    accuracy                           0.61      2185
   macro avg       0.63      0.69      0.59      2185
weighted avg       0.79      0.61      0.64      2185


--- AAPL | Logistic Regression ---
              precision    recall  f1-score   support

    Negative       0.70      0.84      0.77       408
     Neutral       0.60      0.83      0.69       326
    Positive       0.93      0.80      0.86      1451

    accuracy                           0.81      2185
   macro avg       0.74      0.82      0.77      2185
weighted avg       0.84      0.81      0.82      2185


--- AAPL | RoBERTa ---
              precision    recall  f1-score   support

    Negative       0.58      0.23      0.33       408

In [ ]:
# ============================================================
# SECTION 3 — PART 6B: Confusion Matrix
# ============================================================

from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

def plot_confusion_matrices(scored_df, asset_name):
    """
    رسم Confusion Matrix لكل نموذج
    """
    eval_df = scored_df[scored_df['sentiment_label'].notna()].copy()
    y_true  = eval_df['sentiment_label']
    labels  = ['Positive', 'Negative', 'Neutral']

    models = {
        'VADER'               : eval_df['vader_label'],
        'Logistic Regression' : eval_df['finbert_label'],
        'RoBERTa'             : eval_df['roberta_label'],
    }

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=list(models.keys()),
    )

    colors_scale = [
        [0.0, '#0d1117'],
        [0.2, '#1a3a5c'],
        [0.5, '#1f6aa5'],
        [0.8, '#3498db'],
        [1.0, '#58a6ff'],
    ]

    for col_idx, (model_name, y_pred) in enumerate(models.items(), 1):
        cm = confusion_matrix(y_true, y_pred, labels=labels)

        # Normalize بالنسبة المئوية
        cm_norm = cm.astype(float)
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm  = np.divide(cm_norm, row_sums,
                             where=row_sums != 0) * 100

        # نص داخل كل خلية: العدد والنسبة
        text = [[f'{cm[i][j]}<br>{cm_norm[i][j]:.1f}%'
                 for j in range(len(labels))]
                for i in range(len(labels))]

        fig.add_trace(
            go.Heatmap(
                z=cm_norm,
                x=labels,
                y=labels,
                text=text,
                texttemplate='%{text}',
                colorscale=colors_scale,
                showscale=(col_idx == 3),
                zmin=0, zmax=100,
            ),
            row=1, col=col_idx
        )

    # تسميات المحاور
    for col_idx in range(1, 4):
        axis_x = f'xaxis{col_idx}' if col_idx > 1 else 'xaxis'
        axis_y = f'yaxis{col_idx}' if col_idx > 1 else 'yaxis'
        fig.update_layout(**{
            axis_x: dict(title='Predicted', tickfont=dict(size=11)),
            axis_y: dict(title='Actual',    tickfont=dict(size=11)),
        })

    fig.update_layout(
        title=f'{asset_name} — Confusion Matrix (Sentiment Models)',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=420,
        width=1100,
    )
    fig.show()

    # طباعة أرقام تفصيلية
    print(f'\n{"="*55}')
    print(f'  {asset_name} — Confusion Matrix Details')
    print(f'{"="*55}')
    for model_name, y_pred in models.items():
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print(f'\n  {model_name}:')
        print(f'  {"":12} {"Pred Pos":>10} {"Pred Neg":>10} {"Pred Neu":>10}')
        for i, label in enumerate(labels):
            print(f'  {label:12} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}')

plot_confusion_matrices(apple_scored, 'AAPL')
plot_confusion_matrices(gold_scored,  'Gold')


  AAPL — Confusion Matrix Details

  VADER:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            790        134        527
  Negative             39        247        122
  Neutral              11         16        299

  Logistic Regression:
                 Pred Pos   Pred Neg   Pred Neu
  Positive           1162        129        160
  Negative             42        343         23
  Neutral              40         16        270

  RoBERTa:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            698         55        698
  Negative             71         92        245
  Neutral              96         11        219



  Gold — Confusion Matrix Details

  VADER:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            645        160        430
  Negative            126        773        347
  Neutral              25         30        318

  Logistic Regression:
                 Pred Pos   Pred Neg   Pred Neu
  Positive           1027        124         84
  Negative            196        981         69
  Neutral              29         17        327

  RoBERTa:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            147         66       1022
  Negative             22        211       1013
  Neutral               9         14        350


In [ ]:
# ============================================================
# SECTION 3 — PART 7: Visualization
# ============================================================

def plot_sentiment_comparison(metrics, asset_name):
    fig     = go.Figure()
    metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors  = ['#58a6ff', '#3fb950', '#d29922']

    for i, row in metrics.iterrows():
        fig.add_trace(go.Bar(
            name=row['Model'],
            x=metrics_cols,
            y=[row[c] for c in metrics_cols],
            marker_color=colors[i],
            text=[f'{row[c]:.3f}' for c in metrics_cols],
            textposition='outside',
        ))

    fig.update_layout(
        title=f'{asset_name} — Sentiment Model Comparison (vs Ground Truth)',
        barmode='group',
        yaxis=dict(range=[0, 1.15], title='Score'),
        xaxis_title='Metric',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=420,
        legend=dict(bgcolor='#161b22', bordercolor='#30363d'),
    )
    fig.show()

def plot_score_distribution(scored_df, asset_name):
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=['VADER',
                                        'Logistic Regression',
                                        'RoBERTa'])
    score_cols = ['vader_score', 'finbert_score', 'roberta_score']
    colors     = ['#58a6ff', '#3fb950', '#d29922']

    for i, (col, color) in enumerate(zip(score_cols, colors), 1):
        fig.add_trace(go.Histogram(
            x=scored_df[col],
            marker_color=color,
            opacity=0.8, nbinsx=40,
        ), row=1, col=i)

    fig.update_layout(
        title=f'{asset_name} — Sentiment Score Distributions',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=350, showlegend=False,
    )
    fig.show()

plot_sentiment_comparison(apple_sent_metrics, 'AAPL')
plot_sentiment_comparison(gold_sent_metrics,  'Gold')
plot_score_distribution(apple_scored, 'AAPL')
plot_score_distribution(gold_scored,  'Gold')

## 📈 Section 4 — Price Data from Yahoo Finance

Download daily OHLCV data for AAPL and Gold (GC=F).
Date ranges match the news coverage windows.


In [ ]:
# ============================================================
# SECTION 4 — Price Data from Yahoo Finance
# ============================================================

import yfinance as yf

def download_prices(ticker, start, end, name):
    """
    تحميل بيانات الأسعار من Yahoo Finance
    auto_adjust=True يضبط تلقائياً تأثير توزيعات الأرباح وتقسيم الأسهم
    """
    df = yf.download(ticker, start=start, end=end,
                     auto_adjust=True, progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.columns = ['open', 'high', 'low', 'close', 'volume']
    df.index.name = 'date'
    df = df.dropna().sort_index()

    print(f'{name}: {len(df)} trading days '
          f'({df.index.min().date()} → {df.index.max().date()})')
    print(f'  Close range: ${df["close"].min():.2f} → ${df["close"].max():.2f}')
    return df

# AAPL: من 2020-01-01 — نفس فترة ملف apple_cleaned_dataset.csv
aapl_prices = download_prices('AAPL', '2020-01-01', '2026-05-16', 'AAPL')

# Gold: من 2020-01-01 — نفس فترة ملف gold_cleaned_dataset.csv
gold_prices = download_prices('GC=F', '2020-01-01', '2026-05-16', 'Gold')

print()
print('AAPL tail:')
print(aapl_prices.tail(3))
print()
print('Gold tail:')
print(gold_prices.tail(3))

AAPL: 1601 trading days (2020-01-02 → 2026-05-15)
  Close range: $54.16 → $300.23
Gold: 1603 trading days (2020-01-02 → 2026-05-15)
  Close range: $1477.30 → $5318.40

AAPL tail:
                  open        high         low       close    volume
date                                                                
2026-05-13  293.500000  300.920013  293.500000  298.869995  52684300
2026-05-14  299.820007  300.450012  295.380005  298.209991  35324900
2026-05-15  297.899994  303.200012  296.519989  300.230011  54862800

Gold tail:
                   open         high          low        close  volume
date                                                                  
2026-05-13  4722.700195  4722.700195  4679.500000  4697.700195     228
2026-05-14  4678.100098  4678.100098  4650.299805  4678.100098       5
2026-05-15  4615.200195  4615.200195  4524.299805  4555.799805     607


## 🔧 Section 5 — Feature Engineering

Compute all technical indicators and merge daily-aggregated sentiment features.

**Technical indicators:**
SMA (10/20/50), EMA (12/26), MACD, RSI (14), Bollinger Bands,
price change ratios (1/3/7-day), 10-day volatility, High-Low ratio, Volume ratio

**Sentiment features (from the best-performing model — compared below):**
Mean score, std, positive ratio, negative ratio, neutral ratio, article count


In [ ]:
# ============================================================
# SECTION 5 — PART 1: Technical Indicators
# ============================================================

def add_indicators(df):
    df = df.copy()

    # SMA
    for w in (10, 20, 50):
        df[f'sma_{w}'] = df['close'].rolling(w).mean()

    # EMA
    df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
    df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()

    # MACD
    df['macd_line']   = df['ema_12'] - df['ema_26']
    df['macd_signal'] = df['macd_line'].ewm(span=9, adjust=False).mean()
    df['macd_hist']   = df['macd_line'] - df['macd_signal']

    # RSI (14)
    delta    = df['close'].diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=13, min_periods=14).mean()
    avg_loss = loss.ewm(com=13, min_periods=14).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    df['rsi'] = 100 - (100 / (1 + rs))

    # Bollinger Bands (20-day ±2 std)
    mid = df['close'].rolling(20).mean()
    std = df['close'].rolling(20).std()
    df['bb_upper'] = mid + 2 * std
    df['bb_lower'] = mid - 2 * std
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / mid

    # Price change ratios
    df['ret_1d'] = df['close'].pct_change(1)
    df['ret_3d'] = df['close'].pct_change(3)
    df['ret_7d'] = df['close'].pct_change(7)

    # 10-day rolling volatility
    log_ret = np.log(df['close'] / df['close'].shift(1))
    df['volatility_10d'] = log_ret.rolling(10).std()

    # High-Low ratio
    df['hl_ratio'] = (df['high'] - df['low']) / df['close']

    # Volume ratio vs 10-day average
    df['vol_ratio'] = df['volume'] / df['volume'].rolling(10).mean()

    # ── New Features ──────────────────────────────────────────────────────────

    # 1. close_vs_sma50: نسبة السعر للـ SMA50
    # يقيس هل السهم فوق أو تحت المتوسط بعيد المدى
    df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']

    # 2. rsi_trend: اتجاه RSI خلال 3 أيام
    # موجب = RSI يرتفع (زخم صاعد), سالب = RSI ينزل (زخم هابط)
    df['rsi_trend'] = df['rsi'] - df['rsi'].shift(3)

    # 3. volume_spike: هل حجم التداول أعلى من المعتاد؟
    # يكشف أيام النشاط غير العادي
    df['volume_spike'] = (df['volume'] / df['volume'].rolling(20).mean()) - 1

    # 4. price_momentum: زخم السعر — فرق بين SMA10 و SMA50
    # يقيس قوة الاتجاه القصير مقارنة بالطويل
    df['price_momentum'] = (df['sma_10'] - df['sma_50']) / df['sma_50']

    # 5. bb_position: موقع السعر داخل Bollinger Bands (0=أسفل, 1=أعلى)
    bb_range = df['bb_upper'] - df['bb_lower']
    df['bb_position'] = (df['close'] - df['bb_lower']) / bb_range.replace(0, np.nan)

    return df

aapl_prices = add_indicators(aapl_prices)
gold_prices = add_indicators(gold_prices)
print('Indicators added successfully.')
print(f'AAPL features: {aapl_prices.shape[1]}')
print(f'Gold features: {gold_prices.shape[1]}')

Indicators added successfully.
AAPL features: 28
Gold features: 28


In [ ]:
# ============================================================
# SECTION 5 — PART 2: Aggregate Sentiment to Daily Level
# ============================================================

def forward_roll_weekends(df):
    """
    تحويل أخبار نهاية الأسبوع للاثنين التالي
    حتى لا يضيع أي signal من الأخبار
    """
    day    = df['date'].dt.dayofweek
    offset = day.map({5: 2, 6: 1}).fillna(0).astype(int)
    df = df.copy()
    df['date'] = df['date'] + pd.to_timedelta(offset, unit='D')
    return df

def aggregate_daily_sentiment(scored_df):
    """
    تجميع المقالات على مستوى اليوم
    نستخدم كل البيانات (القديمة + الجديدة) للحصول على
    تغطية sentiment أوسع من 2020 إلى 2026
    """
    df = scored_df.copy()
    df = forward_roll_weekends(df)

    # مؤشرات ثنائية لكل نموذج
    for model in ('vader', 'finbert', 'roberta'):
        lbl = f'{model}_label'
        df[f'{model}_pos'] = (df[lbl] == 'Positive').astype(int)
        df[f'{model}_neg'] = (df[lbl] == 'Negative').astype(int)
        df[f'{model}_neu'] = (df[lbl] == 'Neutral').astype(int)

    # Ensemble score = متوسط الثلاثة نماذج
    df['ensemble_score'] = (df['vader_score'] +
                             df['finbert_score'] +
                             df['roberta_score']) / 3

    daily = df.groupby('date').agg(
        vader_score_mean    = ('vader_score',    'mean'),
        finbert_score_mean  = ('finbert_score',  'mean'),
        roberta_score_mean  = ('roberta_score',  'mean'),
        ensemble_score_mean = ('ensemble_score', 'mean'),
        positive_ratio      = ('vader_pos',      'mean'),
        negative_ratio      = ('vader_neg',      'mean'),
        article_count       = ('vader_score',    'count'),
        score_std           = ('ensemble_score', 'std'),
    ).reset_index()

    daily['score_std'] = daily['score_std'].fillna(0)
    daily = daily.set_index('date').sort_index()
    return daily

apple_daily_sent = aggregate_daily_sentiment(apple_scored)
gold_daily_sent  = aggregate_daily_sentiment(gold_scored)

print(f'Apple daily sentiment: {len(apple_daily_sent)} days '
      f'({apple_daily_sent.index.min().date()} → {apple_daily_sent.index.max().date()})')
print(f'Gold  daily sentiment: {len(gold_daily_sent)} days '
      f'({gold_daily_sent.index.min().date()} → {gold_daily_sent.index.max().date()})')

Apple daily sentiment: 1247 days (2020-01-03 → 2026-05-11)
Gold  daily sentiment: 1236 days (2020-01-07 → 2026-05-12)


In [ ]:
# ============================================================
# SECTION 5 — PART 3: Merge Sentiment + Build Target + Lag Features
# ============================================================

def merge_and_build_target(price_df, sentiment_df):
    """
    دمج بيانات السعر مع الـ sentiment على التاريخ:
    - أيام قبل بداية الأخبار → NaN ثم تُحذف بـ dropna()
    - أيام التداول بدون أخبار → forward-fill
    Target = 5-day forward return (أقل ضوضاء من 1-day و 3-day)
    """
    df = price_df.join(sentiment_df, how='left')

    sent_cols       = sentiment_df.columns.tolist()
    first_news_date = sentiment_df.index.min()

    # قبل بداية الأخبار → NaN (سيُحذف بـ dropna)
    df.loc[df.index < first_news_date, sent_cols] = np.nan

    # بعد بداية الأخبار → forward-fill لأيام بدون تغطية
    df.loc[df.index >= first_news_date, sent_cols] = (
        df.loc[df.index >= first_news_date, sent_cols]
        .ffill().bfill()
    )

    # Target = 5-day forward return
    df['target'] = df['close'].pct_change(5).shift(-5) * 100

    # Lag features — technical + sentiment + new features
    for lag in (1, 2, 3, 5):
        df[f'ret_lag_{lag}']       = df['ret_1d'].shift(lag)
        df[f'rsi_lag_{lag}']       = df['rsi'].shift(lag)
        df[f'macd_lag_{lag}']      = df['macd_line'].shift(lag)
        df[f'sent_lag_{lag}']      = df['ensemble_score_mean'].shift(lag)
        df[f'momentum_lag_{lag}']  = df['price_momentum'].shift(lag)
        df[f'vol_spike_lag_{lag}'] = df['volume_spike'].shift(lag)

    df = df.dropna()
    return df

aapl_ds = merge_and_build_target(aapl_prices, apple_daily_sent)
gold_ds  = merge_and_build_target(gold_prices, gold_daily_sent)

print(f'AAPL dataset: {aapl_ds.shape}')
print(f'Gold dataset: {gold_ds.shape}')
print(f'\nAAPL date range: {aapl_ds.index.min().date()} → {aapl_ds.index.max().date()}')
print(f'Gold date range: {gold_ds.index.min().date()} → {gold_ds.index.max().date()}')
print(f'\nAAPL target stats (5-day forward return %):')
print(aapl_ds['target'].describe().round(4))
print(f'\nGold target stats (5-day forward return %):')
print(gold_ds['target'].describe().round(4))

AAPL dataset: (1542, 61)
Gold dataset: (1544, 61)

AAPL date range: 2020-03-20 → 2026-05-08
Gold date range: 2020-03-20 → 2026-05-08

AAPL target stats (5-day forward return %):
count    1542.0000
mean        0.6120
std         4.0425
min       -22.7474
25%        -1.8813
50%         0.5891
75%         3.0158
max        18.4141
Name: target, dtype: float64

Gold target stats (5-day forward return %):
count    1544.0000
mean        0.3770
std         2.4219
min       -12.0316
25%        -1.0231
50%         0.4338
75%         1.8085
max        10.7074
Name: target, dtype: float64


## 🤖 Section 6 — Price Prediction: XGBoost vs Linear Regression vs Random Forest

**Protocol:**
- Chronological 80/20 split — no shuffling (preserves time order)
- TimeSeriesSplit(5) for cross-validation
- StandardScaler applied only to Linear Regression
- Evaluated on R², RMSE, MAE, MAPE


In [ ]:
# ============================================================
# SECTION 6 — PART 1: Helper Functions & Feature Definition
# ============================================================

from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from sklearn.preprocessing   import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics         import r2_score, mean_squared_error, mean_absolute_error
from xgboost                 import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

AAPL_TOP_FEATURES = [
    # Technical
    'macd_lag_2', 'volatility_10d', 'macd_hist',
    'rsi', 'ret_7d', 'rsi_lag_1', 'bb_width',
    'macd_line', 'rsi_lag_2', 'rsi_lag_5',
    # Sentiment
    'roberta_score_mean', 'positive_ratio',
    'vader_score_mean', 'negative_ratio',
    'ensemble_score_mean',
    # New Features
    'close_vs_sma50', 'rsi_trend', 'volume_spike',
    'price_momentum', 'bb_position',
    'momentum_lag_1', 'momentum_lag_3',
    'vol_spike_lag_1', 'vol_spike_lag_3',
]

GOLD_TOP_FEATURES = [
    # Technical
    'negative_ratio', 'bb_width', 'macd_hist',
    'macd_lag_5', 'rsi_lag_5', 'vol_ratio',
    'macd_line', 'rsi_lag_2', 'rsi_lag_3',
    # Sentiment
    'vader_score_mean', 'positive_ratio',
    'score_std', 'finbert_score_mean',
    'sent_lag_2', 'ret_lag_5',
    # New Features
    'close_vs_sma50', 'rsi_trend', 'volume_spike',
    'price_momentum', 'bb_position',
    'momentum_lag_1', 'momentum_lag_3',
    'vol_spike_lag_1', 'vol_spike_lag_3',
]

TECHNICAL_ONLY_FEATURES = [
    'ret_1d', 'ret_3d', 'ret_7d',
    'volatility_10d', 'hl_ratio', 'vol_ratio',
    'bb_width', 'rsi', 'macd_line', 'macd_signal', 'macd_hist',
    'ret_lag_1', 'ret_lag_2', 'ret_lag_3', 'ret_lag_5',
    'rsi_lag_1', 'rsi_lag_2', 'rsi_lag_3', 'rsi_lag_5',
    'macd_lag_1', 'macd_lag_2', 'macd_lag_3', 'macd_lag_5',
    # New Features
    'close_vs_sma50', 'rsi_trend', 'volume_spike',
    'price_momentum', 'bb_position',
    'momentum_lag_1', 'momentum_lag_3',
    'vol_spike_lag_1', 'vol_spike_lag_3',
]

def get_features(df, asset='aapl'):
    feat_list = AAPL_TOP_FEATURES if asset == 'aapl' else GOLD_TOP_FEATURES
    return [c for c in feat_list if c in df.columns]

def directional_accuracy(y_true, y_pred):
    correct = np.sign(y_true) == np.sign(y_pred)
    return round(float(correct.mean()) * 100, 2)

def regression_metrics(y_true, y_pred, name):
    return {
        'Model'    : name,
        'R²'       : round(r2_score(y_true, y_pred), 4),
        'RMSE'     : round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE'      : round(mean_absolute_error(y_true, y_pred), 4),
        'Dir Acc%' : directional_accuracy(y_true, y_pred),
    }

def ts_cv_r2(model, X, y, n_splits=5):
    tscv   = TimeSeriesSplit(n_splits=n_splits)
    scores = [r2_score(y[v], model.fit(X[t], y[t]).predict(X[v]))
              for t, v in tscv.split(X)]
    return round(float(np.mean(scores)), 4)

print('Helper functions ready.')

Helper functions ready.


In [ ]:
# ============================================================
# SECTION 6 — PART 2: Train XGBoost
# ============================================================

def train_xgboost(X_tr, y_tr, X_te, y_te):
    xgb = XGBRegressor(
        n_estimators          = 500,
        max_depth             = 4,
        learning_rate         = 0.03,
        subsample             = 0.8,
        colsample_bytree      = 0.7,
        min_child_weight      = 5,
        reg_alpha             = 0.3,
        reg_lambda            = 2.0,
        early_stopping_rounds = 40,
        eval_metric           = 'rmse',
        random_state          = 42,
        verbosity             = 0,
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    pred = xgb.predict(X_te)
    cv   = ts_cv_r2(
        XGBRegressor(n_estimators=300, max_depth=4,
                     learning_rate=0.03, random_state=42, verbosity=0),
        X_tr, y_tr
    )
    m = regression_metrics(y_te, pred, 'XGBoost')
    m['CV R²'] = cv
    print(f'XGBoost       → R²: {m["R²"]}  RMSE: {m["RMSE"]}  '
          f'Dir Acc: {m["Dir Acc%"]}%  CV R²: {cv}')
    return {'metrics': m, 'model': xgb, 'pred': pred}

print('XGBoost trainer ready.')

XGBoost trainer ready.


In [ ]:
# ============================================================
# SECTION 6 — PART 3: Train Linear Regression (Technical Only)
# ============================================================

def train_linear_regression(ds, y_tr, y_te):
    """
    Linear Regression على Technical Indicators فقط — بدون sentiment
    للمقارنة وقياس تأثير الأخبار على النماذج الأخرى
    """
    tech_feat = [c for c in TECHNICAL_ONLY_FEATURES if c in ds.columns]
    split     = len(y_tr)

    X_tr_tech = ds[tech_feat].values[:split]
    X_te_tech = ds[tech_feat].values[split:split + len(y_te)]

    scaler  = RobustScaler()
    X_tr_sc = scaler.fit_transform(X_tr_tech)
    X_te_sc = scaler.transform(X_te_tech)

    lr = LinearRegression()
    lr.fit(X_tr_sc, y_tr)
    pred = lr.predict(X_te_sc)
    cv   = ts_cv_r2(LinearRegression(), X_tr_sc, y_tr)

    m = regression_metrics(y_te, pred, 'Linear Regression')
    m['CV R²'] = cv
    print(f'Linear Reg    → R²: {m["R²"]}  RMSE: {m["RMSE"]}  '
          f'Dir Acc: {m["Dir Acc%"]}%  CV R²: {cv}')
    print(f'  (Technical indicators only — no sentiment)')
    return {
        'metrics' : m,
        'model'   : lr,
        'scaler'  : scaler,
        'pred'    : pred,
        'feat'    : tech_feat,
    }

print('Linear Regression trainer ready.')

Linear Regression trainer ready.


In [ ]:
# ============================================================
# SECTION 6 — PART 4: Train Random Forest
# ============================================================

def train_random_forest(X_tr, y_tr, X_te, y_te):
    rf = RandomForestRegressor(
        n_estimators     = 500,
        max_features     = 'sqrt',
        max_depth        = 5,
        min_samples_leaf = 4,
        min_samples_split= 8,
        random_state     = 42,
        n_jobs           = -1,
    )
    rf.fit(X_tr, y_tr)
    pred = rf.predict(X_te)
    cv   = ts_cv_r2(
        RandomForestRegressor(n_estimators=200, max_depth=5,
                              min_samples_leaf=4, random_state=42,
                              n_jobs=-1),
        X_tr, y_tr
    )
    m = regression_metrics(y_te, pred, 'Random Forest')
    m['CV R²'] = cv
    print(f'Random Forest → R²: {m["R²"]}  RMSE: {m["RMSE"]}  '
          f'Dir Acc: {m["Dir Acc%"]}%  CV R²: {cv}')
    return {'metrics': m, 'model': rf, 'pred': pred}

print('Random Forest trainer ready.')

Random Forest trainer ready.


In [ ]:
# ============================================================
# SECTION 6 — PART 5: Run All Models on AAPL & Gold
# ============================================================

def run_all_models(ds, asset_name):
    feat   = get_features(ds, asset=asset_name.lower())
    X_all  = ds[feat].values
    y      = ds['target'].values
    dates  = ds.index
    split  = int(len(X_all) * 0.8)

    X_tr, X_te = X_all[:split], X_all[split:]
    y_tr, y_te = y[:split], y[split:]
    dates_te   = dates[split:]

    print(f'\n=== {asset_name} — Train: {len(X_tr)} | Test: {len(X_te)} ===')
    print(f'Features used ({len(feat)}): {feat}')

    results   = {}
    all_preds = {'dates': dates_te, 'y_true': y_te}

    # XGBoost — مع sentiment + new features
    results['XGBoost'] = train_xgboost(X_tr, y_tr, X_te, y_te)
    all_preds['XGBoost'] = results['XGBoost']['pred']

    # Linear Regression — technical + new features فقط
    results['Linear Regression'] = train_linear_regression(ds, y_tr, y_te)
    all_preds['Linear Regression'] = results['Linear Regression']['pred']

    # Random Forest — مع sentiment + new features
    results['Random Forest'] = train_random_forest(X_tr, y_tr, X_te, y_te)
    all_preds['Random Forest'] = results['Random Forest']['pred']

    tbl = pd.DataFrame([r['metrics'] for r in results.values()])
    print(f'\n{"="*65}')
    print(f'  {asset_name} — Final Comparison Table')
    print(f'{"="*65}')
    print(tbl.to_string(index=False))

    return results, all_preds, feat

aapl_results, aapl_preds, aapl_feat = run_all_models(aapl_ds, 'aapl')
gold_results, gold_preds, gold_feat = run_all_models(gold_ds, 'gold')


=== aapl — Train: 1233 | Test: 309 ===
Features used (24): ['macd_lag_2', 'volatility_10d', 'macd_hist', 'rsi', 'ret_7d', 'rsi_lag_1', 'bb_width', 'macd_line', 'rsi_lag_2', 'rsi_lag_5', 'roberta_score_mean', 'positive_ratio', 'vader_score_mean', 'negative_ratio', 'ensemble_score_mean', 'close_vs_sma50', 'rsi_trend', 'volume_spike', 'price_momentum', 'bb_position', 'momentum_lag_1', 'momentum_lag_3', 'vol_spike_lag_1', 'vol_spike_lag_3']
XGBoost       → R²: 0.0655  RMSE: 4.2832  Dir Acc: 56.63%  CV R²: -0.7984
Linear Reg    → R²: -0.0458  RMSE: 4.531  Dir Acc: 50.81%  CV R²: -0.6991
  (Technical indicators only — no sentiment)
Random Forest → R²: 0.0477  RMSE: 4.3238  Dir Acc: 51.78%  CV R²: -0.4096

  aapl — Final Comparison Table
            Model      R²   RMSE    MAE  Dir Acc%   CV R²
          XGBoost  0.0655 4.2832 3.1393     56.63 -0.7984
Linear Regression -0.0458 4.5310 3.2892     50.81 -0.6991
    Random Forest  0.0477 4.3238 3.1809     51.78 -0.4096

=== gold — Train: 1235 | 

In [ ]:
import os
import joblib
from google.colab import drive

# ── تحميل Google Drive ───────────────────────────────────────
drive.mount('/content/drive')

# ── مسار الحفظ داخل Drive ────────────────────────────────────
folder_name    = "XGBoost_Models_Project"
project_folder = f"/content/drive/MyDrive/{folder_name}"

if not os.path.exists(project_folder):
    os.makedirs(project_folder)
    print(f"✅ تم إنشاء المجلد: {project_folder}")
else:
    print(f"📁 المجلد موجود: {project_folder}")

# ── استخراج النماذج من results ────────────────────────────────
xgb_aapl = aapl_results['XGBoost']['model']
xgb_gold = gold_results['XGBoost']['model']

# ── مسارات الحفظ ─────────────────────────────────────────────
apple_model_path = os.path.join(project_folder, 'xgb_apple.pkl')
gold_model_path  = os.path.join(project_folder, 'xgb_gold.pkl')
aapl_feat_path   = os.path.join(project_folder, 'aapl_features.pkl')
gold_feat_path   = os.path.join(project_folder, 'gold_features.pkl')

# ── حفظ النماذج والـ features ────────────────────────────────
joblib.dump(xgb_aapl,  apple_model_path)
joblib.dump(xgb_gold,  gold_model_path)
joblib.dump(aapl_feat, aapl_feat_path)
joblib.dump(gold_feat, gold_feat_path)

print('=' * 50)
print('  XGBoost Models Saved to Google Drive')
print('=' * 50)
print(f'  📍 Location : MyDrive/{folder_name}/')
print(f'  🍏 AAPL model    : xgb_apple.pkl')
print(f'  🏆 Gold model    : xgb_gold.pkl')
print(f'  📋 AAPL features : aapl_features.pkl')
print(f'  📋 Gold features : gold_features.pkl')

# ── تحقق من الحفظ ────────────────────────────────────────────
loaded_aapl = joblib.load(apple_model_path)
loaded_gold = joblib.load(gold_model_path)
print(f'\n  Verification:')
print(f'  AAPL model features : {loaded_aapl.n_features_in_}')
print(f'  Gold model features : {loaded_gold.n_features_in_}')
print('=' * 50)

Mounted at /content/drive
✅ تم إنشاء المجلد: /content/drive/MyDrive/XGBoost_Models_Project
  XGBoost Models Saved to Google Drive
  📍 Location : MyDrive/XGBoost_Models_Project/
  🍏 AAPL model    : xgb_apple.pkl
  🏆 Gold model    : xgb_gold.pkl
  📋 AAPL features : aapl_features.pkl
  📋 Gold features : gold_features.pkl

  Verification:
  AAPL model features : 24
  Gold model features : 24


In [ ]:
import os
import joblib

# ── مسار الحفظ ───────────────────────────────────────────────
onedrive_path = os.path.expanduser(r"~\OneDrive")
folder_name   = "XGBoost_Models_Project"
project_folder = os.path.join(onedrive_path, folder_name)

if not os.path.exists(project_folder):
    os.makedirs(project_folder)
    print(f"✅ تم إنشاء المجلد: {project_folder}")
else:
    print(f"📁 المجلد موجود: {project_folder}")

# ── استخراج النماذج من results ────────────────────────────────
xgb_aapl = aapl_results['XGBoost']['model']
xgb_gold = gold_results['XGBoost']['model']

# ── حفظ النماذج ──────────────────────────────────────────────
apple_model_path = os.path.join(project_folder, 'xgb_apple.pkl')
gold_model_path  = os.path.join(project_folder, 'xgb_gold.pkl')

joblib.dump(xgb_aapl, apple_model_path)
print(f"🍏 تم حفظ نموذج Apple: {apple_model_path}")

joblib.dump(xgb_gold, gold_model_path)
print(f"🏆 تم حفظ نموذج Gold: {gold_model_path}")

# ── تحقق من الحفظ ────────────────────────────────────────────
loaded_aapl = joblib.load(apple_model_path)
loaded_gold = joblib.load(gold_model_path)
print(f"\n✅ تم التحقق — AAPL features: {loaded_aapl.n_features_in_}")
print(f"✅ تم التحقق — Gold features: {loaded_gold.n_features_in_}")

📁 المجلد موجود: ~\OneDrive/XGBoost_Models_Project
🍏 تم حفظ نموذج Apple: ~\OneDrive/XGBoost_Models_Project/xgb_apple.pkl
🏆 تم حفظ نموذج Gold: ~\OneDrive/XGBoost_Models_Project/xgb_gold.pkl

✅ تم التحقق — AAPL features: 24
✅ تم التحقق — Gold features: 24


## 📊 Section 7 — Visualisations

In [ ]:
# ============================================================
# SECTION 7 — PART 1: Predicted vs Actual Return Chart
# ============================================================

def plot_predictions(preds, asset_name):
    fig    = go.Figure()
    colors = {'XGBoost'          : '#58a6ff',
              'Linear Regression' : '#3fb950',
              'Random Forest'     : '#d29922'}

    fig.add_trace(go.Scatter(
        x=preds['dates'], y=preds['y_true'],
        name='Actual Return',
        line=dict(color='#e6edf3', width=2)))

    for model, color in colors.items():
        if model in preds:
            fig.add_trace(go.Scatter(
                x=preds['dates'], y=preds[model], name=model,
                line=dict(color=color, width=1.5, dash='dot')))

    fig.update_layout(
        title=f'{asset_name} — Predicted vs Actual 5-Day Return (%)',
        xaxis_title='Date', yaxis_title='Return (%)',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=420,
        legend=dict(bgcolor='#161b22', bordercolor='#30363d'),
    )
    fig.show()

plot_predictions(aapl_preds, 'AAPL')
plot_predictions(gold_preds,  'Gold')

In [ ]:
# ============================================================
# SECTION 7 — PART 2: Model Comparison Bar Chart
# ============================================================

def plot_regression_comparison(results, asset_name):
    tbl     = pd.DataFrame([r['metrics'] for r in results.values()])
    models  = tbl['Model'].tolist()
    colors  = ['#58a6ff', '#3fb950', '#d29922']
    metrics = ['R²', 'RMSE', 'MAE', 'Dir Acc%']

    fig = make_subplots(rows=1, cols=4, subplot_titles=metrics)
    for i, metric in enumerate(metrics, 1):
        fig.add_trace(go.Bar(
            x=models, y=tbl[metric],
            marker_color=colors,
            text=[f'{v:.3f}' for v in tbl[metric]],
            textposition='outside',
            showlegend=False,
        ), row=1, col=i)

    fig.update_layout(
        title=f'{asset_name} — Regression Model Comparison',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=420,
    )
    fig.show()

plot_regression_comparison(aapl_results, 'AAPL')
plot_regression_comparison(gold_results,  'Gold')

In [ ]:
# ============================================================
# SECTION 7 — PART 3: XGBoost Feature Importance
# ============================================================

def plot_feature_importance(results, feat_cols, asset_name, top_n=15):
    fi = pd.Series(
        results['XGBoost']['model'].feature_importances_,
        index=feat_cols
    ).sort_values(ascending=True).tail(top_n)

    fig = go.Figure(go.Bar(
        x=fi.values, y=fi.index,
        orientation='h',
        marker_color='#58a6ff',
    ))
    fig.update_layout(
        title=f'{asset_name} — XGBoost Top {top_n} Feature Importances',
        xaxis_title='Importance',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=500,
    )
    fig.show()

plot_feature_importance(aapl_results, aapl_feat, 'AAPL')
plot_feature_importance(gold_results,  gold_feat, 'Gold')

In [ ]:
# ============================================================
# SECTION 7 — PART 4: Price Chart with SMA + Bollinger Bands
# ============================================================

def plot_price_chart(ds, asset_name):
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        row_heights=[0.6, 0.2, 0.2],
                        subplot_titles=[
                            f'{asset_name} Price + SMA + Bollinger Bands',
                            'RSI (14)',
                            'MACD'])

    fig.add_trace(go.Scatter(x=ds.index, y=ds['close'],
                             name='Close',
                             line=dict(color='#e6edf3', width=1.5)),
                  row=1, col=1)
    for sma, color in [('sma_10','#58a6ff'),
                        ('sma_20','#d29922'),
                        ('sma_50','#f0883e')]:
        if sma in ds.columns:
            fig.add_trace(go.Scatter(x=ds.index, y=ds[sma],
                                      name=sma.upper(),
                                      line=dict(color=color, width=1)),
                          row=1, col=1)
    fig.add_trace(go.Scatter(x=ds.index, y=ds['bb_upper'],
                              name='BB Upper',
                              line=dict(color='rgba(63,185,80,0.4)', width=1),
                              fill=None), row=1, col=1)
    fig.add_trace(go.Scatter(x=ds.index, y=ds['bb_lower'],
                              name='BB Lower',
                              line=dict(color='rgba(63,185,80,0.4)', width=1),
                              fill='tonexty',
                              fillcolor='rgba(63,185,80,0.05)'),
                  row=1, col=1)

    fig.add_trace(go.Scatter(x=ds.index, y=ds['rsi'], name='RSI',
                              line=dict(color='#bc8cff', width=1.5)),
                  row=2, col=1)
    fig.add_hline(y=70, line_dash='dash',
                  line_color='rgba(248,81,73,0.7)', row=2, col=1)
    fig.add_hline(y=30, line_dash='dash',
                  line_color='rgba(63,185,80,0.7)', row=2, col=1)

    hist_colors = ['rgba(63,185,80,0.6)' if v >= 0 else 'rgba(248,81,73,0.6)'
                   for v in ds['macd_hist'].fillna(0)]
    fig.add_trace(go.Bar(x=ds.index, y=ds['macd_hist'],
                          name='Hist', marker_color=hist_colors),
                  row=3, col=1)
    fig.add_trace(go.Scatter(x=ds.index, y=ds['macd_line'], name='MACD',
                              line=dict(color='#58a6ff', width=1.2)),
                  row=3, col=1)
    fig.add_trace(go.Scatter(x=ds.index, y=ds['macd_signal'], name='Signal',
                              line=dict(color='#f0883e', width=1.2)),
                  row=3, col=1)

    fig.update_layout(
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=700,
        legend=dict(bgcolor='#161b22', bordercolor='#30363d'),
    )
    fig.show()

plot_price_chart(aapl_ds, 'AAPL')
plot_price_chart(gold_ds,  'Gold')

In [ ]:
# ============================================================
# SECTION 7 — PART 5: Sentiment Timeline vs Price
# ============================================================

def plot_sentiment_timeline(ds, asset_name):
    if 'ensemble_score_mean' not in ds.columns:
        print(f'{asset_name}: No sentiment data — skipping.')
        return

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=[f'{asset_name} Close Price',
                                        'Daily Ensemble Sentiment Score'])

    fig.add_trace(go.Scatter(x=ds.index, y=ds['close'],
                             name='Close',
                             line=dict(color='#58a6ff', width=1.5)),
                  row=1, col=1)

    colors = ['rgba(63,185,80,0.8)' if v >= 0 else 'rgba(248,81,73,0.8)'
              for v in ds['ensemble_score_mean'].fillna(0)]
    fig.add_trace(go.Bar(x=ds.index, y=ds['ensemble_score_mean'],
                          marker_color=colors, name='Sentiment'),
                  row=2, col=1)

    fig.update_layout(
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=500,
    )
    fig.show()

plot_sentiment_timeline(aapl_ds, 'AAPL')
plot_sentiment_timeline(gold_ds,  'Gold')

In [ ]:
# ============================================================
# SECTION 7 — PART 6: BUY / HOLD / SELL Signal Chart
# ============================================================

def generate_signals(ds, results, asset_name):
    feat  = aapl_feat if asset_name == 'AAPL' else gold_feat
    X     = ds[feat].values
    xgb   = results['XGBoost']['model']
    preds = xgb.predict(X)

    pred_75 = np.percentile(preds, 75)
    pred_25 = np.percentile(preds, 25)

    has_sentiment = 'ensemble_score_mean' in ds.columns

    scores = []
    for i in range(len(ds)):
        score = 0

        # Signal 1: XGBoost
        if preds[i] >= pred_75:
            score += 1
        elif preds[i] <= pred_25:
            score -= 1

        # Signal 2: RSI
        rsi_val = ds['rsi'].iloc[i]
        if rsi_val < 35:
            score += 1
        elif rsi_val > 65:
            score -= 1

        # Signal 3: Sentiment
        if has_sentiment:
            sent_val = ds['ensemble_score_mean'].iloc[i]
            if sent_val > 0.05:
                score += 1
            elif sent_val < -0.05:
                score -= 1

        scores.append(score)

    ds = ds.copy()
    ds['signal_score'] = scores
    ds['signal'] = ds['signal_score'].apply(
        lambda s: 'BUY' if s >= 2 else ('SELL' if s <= -2 else 'HOLD')
    )
    return ds


def plot_signals(ds, asset_name):
    ds = generate_signals(ds,
                          aapl_results if asset_name == 'AAPL' else gold_results,
                          asset_name)

    buy_days  = ds[ds['signal'] == 'BUY']
    sell_days = ds[ds['signal'] == 'SELL']
    hold_days = ds[ds['signal'] == 'HOLD']

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.7, 0.3],
                        subplot_titles=[
                            f'{asset_name} — BUY / HOLD / SELL Signals',
                            'Signal Score'])

    fig.add_trace(go.Scatter(
        x=ds.index, y=ds['close'],
        name='Close Price',
        line=dict(color='#e6edf3', width=1.5)),
        row=1, col=1)

    fig.add_trace(go.Scatter(
        x=buy_days.index, y=buy_days['close'],
        name='BUY', mode='markers',
        marker=dict(symbol='triangle-up', size=10,
                    color='#3fb950',
                    line=dict(width=1, color='#ffffff'))),
        row=1, col=1)

    fig.add_trace(go.Scatter(
        x=sell_days.index, y=sell_days['close'],
        name='SELL', mode='markers',
        marker=dict(symbol='triangle-down', size=10,
                    color='#f85149',
                    line=dict(width=1, color='#ffffff'))),
        row=1, col=1)

    fig.add_trace(go.Scatter(
        x=hold_days.index, y=hold_days['close'],
        name='HOLD', mode='markers',
        marker=dict(symbol='circle', size=4,
                    color='rgba(210,153,34,0.4)')),
        row=1, col=1)

    bar_colors = [
        '#3fb950' if s >= 2 else ('#f85149' if s <= -2 else '#d29922')
        for s in ds['signal_score']
    ]
    fig.add_trace(go.Bar(
        x=ds.index, y=ds['signal_score'],
        name='Score', marker_color=bar_colors,
        showlegend=False),
        row=2, col=1)

    fig.add_hline(y=2,  line_dash='dash',
                  line_color='rgba(63,185,80,0.6)',  row=2, col=1)
    fig.add_hline(y=-2, line_dash='dash',
                  line_color='rgba(248,81,73,0.6)',  row=2, col=1)
    fig.add_hline(y=0,  line_dash='dot',
                  line_color='rgba(255,255,255,0.2)', row=2, col=1)

    total      = len(ds)
    buy_count  = len(buy_days)
    sell_count = len(sell_days)
    hold_count = len(hold_days)

    fig.update_layout(
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=600,
        legend=dict(bgcolor='#161b22', bordercolor='#30363d'),
        annotations=[dict(
            x=0.01, y=1.02, xref='paper', yref='paper',
            text=(f'BUY: {buy_count} days ({buy_count/total*100:.1f}%)  |  '
                  f'HOLD: {hold_count} days ({hold_count/total*100:.1f}%)  |  '
                  f'SELL: {sell_count} days ({sell_count/total*100:.1f}%)'),
            showarrow=False,
            font=dict(size=12, color='#8b949e'),
        )]
    )
    fig.show()

    last          = ds.iloc[-1]
    has_sentiment = 'ensemble_score_mean' in ds.columns
    sent_display  = f'{last["ensemble_score_mean"]:.3f}' if has_sentiment else 'N/A'
    max_score     = 3 if has_sentiment else 2

    print(f'\n{"="*45}')
    print(f'  {asset_name} — LATEST SIGNAL: {last["signal"]}')
    print(f'{"="*45}')
    print(f'  Date            : {ds.index[-1].date()}')
    print(f'  Close Price     : {last["close"]:.2f}')
    print(f'  RSI             : {last["rsi"]:.1f}')
    print(f'  Sentiment Score : {sent_display}')
    print(f'  Signal Score    : {int(last["signal_score"])} / {max_score}')
    print(f'{"="*45}\n')


plot_signals(aapl_ds, 'AAPL')
plot_signals(gold_ds,  'Gold')


  AAPL — LATEST SIGNAL: HOLD
  Date            : 2026-05-08
  Close Price     : 293.05
  RSI             : 72.9
  Sentiment Score : 0.302
  Signal Score    : -1 / 3




  Gold — LATEST SIGNAL: HOLD
  Date            : 2026-05-08
  Close Price     : 4720.40
  RSI             : 51.9
  Sentiment Score : 0.315
  Signal Score    : 1 / 3



## 📋 Section 8 — Final Summary Tables

Clean side-by-side comparison of all models for easy reporting.


In [ ]:
# ============================================================
# SECTION 8 — PART 1: Final Sentiment Comparison Table
# ============================================================

print('=' * 65)
print('  SENTIMENT MODEL COMPARISON — AAPL (vs Ground Truth Labels)')
print('=' * 65)
print(apple_sent_metrics.to_string(index=False))

print()
print('=' * 65)
print('  SENTIMENT MODEL COMPARISON — GOLD (vs Ground Truth Labels)')
print('=' * 65)
print(gold_sent_metrics.to_string(index=False))

  SENTIMENT MODEL COMPARISON — AAPL (vs Ground Truth Labels)
              Model  Accuracy  Precision  Recall  F1-Score
              VADER    0.6114     0.7878  0.6114    0.6426
Logistic Regression    0.8124     0.8405  0.8124    0.8190
            RoBERTa    0.4618     0.6727  0.4618    0.5049

  SENTIMENT MODEL COMPARISON — GOLD (vs Ground Truth Labels)
              Model  Accuracy  Precision  Recall  F1-Score
              VADER    0.6083     0.7390  0.6083    0.6370
Logistic Regression    0.8181     0.8257  0.8181    0.8193
            RoBERTa    0.2481     0.6931  0.2481    0.2431


In [ ]:
# ============================================================
# SECTION 8 — PART 2: Final Price Prediction Comparison Table
# ============================================================

for asset_name, results in [('AAPL', aapl_results), ('Gold', gold_results)]:
    tbl = pd.DataFrame([r['metrics'] for r in results.values()])
    print('=' * 65)
    print(f'  PRICE PREDICTION MODEL COMPARISON — {asset_name}')
    print('=' * 65)
    print(tbl.to_string(index=False))
    print()

  PRICE PREDICTION MODEL COMPARISON — AAPL
            Model      R²   RMSE    MAE  Dir Acc%   CV R²
          XGBoost  0.0655 4.2832 3.1393     56.63 -0.7984
Linear Regression -0.0458 4.5310 3.2892     50.81 -0.6991
    Random Forest  0.0476 4.3238 3.1813     51.78 -0.4168

  PRICE PREDICTION MODEL COMPARISON — Gold
            Model      R²   RMSE    MAE  Dir Acc%   CV R²
          XGBoost -0.0158 3.4462 2.6360     58.58 -0.3761
Linear Regression -0.4784 4.1575 3.0007     58.90 -0.2981
    Random Forest -0.0353 3.4791 2.6466     57.61 -0.1537



In [ ]:
# ============================================================
# SECTION 8 — PART 3: Final Sentiment Bar Chart
# ============================================================

def plot_sentiment_final(metrics, asset_name):
    fig          = go.Figure()
    metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors       = ['#58a6ff', '#3fb950', '#d29922']

    for i, row in metrics.iterrows():
        fig.add_trace(go.Bar(
            name=row['Model'],
            x=metrics_cols,
            y=[row[c] for c in metrics_cols],
            marker_color=colors[i],
            text=[f'{row[c]:.3f}' for c in metrics_cols],
            textposition='outside',
        ))

    fig.update_layout(
        title=f'{asset_name} — Sentiment Model Comparison (vs Ground Truth)',
        barmode='group',
        yaxis=dict(range=[0, 1.15], title='Score'),
        xaxis_title='Metric',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=420,
        legend=dict(bgcolor='#161b22', bordercolor='#30363d'),
    )
    fig.show()

plot_sentiment_final(apple_sent_metrics, 'AAPL')
plot_sentiment_final(gold_sent_metrics,  'Gold')

In [ ]:
# ============================================================
# SECTION 8 — PART 4: Final Price Prediction Bar Chart
# ============================================================

def plot_price_final(results, asset_name):
    tbl     = pd.DataFrame([r['metrics'] for r in results.values()])
    models  = tbl['Model'].tolist()
    colors  = ['#58a6ff', '#3fb950', '#d29922']
    metrics = ['R²', 'RMSE', 'MAE', 'Dir Acc%']

    fig = make_subplots(rows=1, cols=4, subplot_titles=metrics)
    for i, metric in enumerate(metrics, 1):
        fig.add_trace(go.Bar(
            x=models, y=tbl[metric],
            marker_color=colors,
            text=[f'{v:.3f}' for v in tbl[metric]],
            textposition='outside',
            showlegend=False,
        ), row=1, col=i)

    fig.update_layout(
        title=f'{asset_name} — Price Prediction Model Comparison',
        plot_bgcolor='#0d1117', paper_bgcolor='#161b22',
        font_color='#e6edf3', height=420,
    )
    fig.show()

plot_price_final(aapl_results, 'AAPL')
plot_price_final(gold_results,  'Gold')

In [ ]:
# ============================================================
# SECTION 8 — PART 5: Confusion Matrix
# ============================================================

plot_confusion_matrices(apple_scored, 'AAPL')
plot_confusion_matrices(gold_scored,  'Gold')


  AAPL — Confusion Matrix Details

  VADER:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            790        134        527
  Negative             39        247        122
  Neutral              11         16        299

  Logistic Regression:
                 Pred Pos   Pred Neg   Pred Neu
  Positive           1162        129        160
  Negative             42        343         23
  Neutral              40         16        270

  RoBERTa:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            698         55        698
  Negative             71         92        245
  Neutral              96         11        219



  Gold — Confusion Matrix Details

  VADER:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            645        160        430
  Negative            126        773        347
  Neutral              25         30        318

  Logistic Regression:
                 Pred Pos   Pred Neg   Pred Neu
  Positive           1027        124         84
  Negative            196        981         69
  Neutral              29         17        327

  RoBERTa:
                 Pred Pos   Pred Neg   Pred Neu
  Positive            147         66       1022
  Negative             22        211       1013
  Neutral               9         14        350


In [ ]:
# ============================================================
# SECTION 8 — PART 6: Latest BUY / HOLD / SELL Summary
# ============================================================

for asset_name, ds, results in [('AAPL', aapl_ds, aapl_results),
                                  ('Gold', gold_ds, gold_results)]:

    ds_sig = generate_signals(ds, results, asset_name)
    last   = ds_sig.iloc[-1]
    total  = len(ds_sig)

    buy_count  = (ds_sig['signal'] == 'BUY').sum()
    sell_count = (ds_sig['signal'] == 'SELL').sum()
    hold_count = (ds_sig['signal'] == 'HOLD').sum()

    has_sentiment = 'ensemble_score_mean' in ds_sig.columns
    sent_display  = f'{last["ensemble_score_mean"]:.3f}' if has_sentiment else 'N/A'
    max_score     = 3 if has_sentiment else 2

    print('=' * 55)
    print(f'  {asset_name} — FINAL INVESTMENT SUMMARY')
    print('=' * 55)
    print(f'  Latest Date      : {ds_sig.index[-1].date()}')
    print(f'  Close Price      : {last["close"]:.2f}')
    print(f'  RSI              : {last["rsi"]:.1f}')
    print(f'  Sentiment Score  : {sent_display}')
    print(f'  Signal Score     : {int(last["signal_score"])} / {max_score}')
    print(f'  RECOMMENDATION   : *** {last["signal"]} ***')
    print(f'  ─────────────────────────────────────────')
    print(f'  BUY  days : {buy_count:4d} ({buy_count/total*100:.1f}%)')
    print(f'  HOLD days : {hold_count:4d} ({hold_count/total*100:.1f}%)')
    print(f'  SELL days : {sell_count:4d} ({sell_count/total*100:.1f}%)')
    print('=' * 55)
    print()

  AAPL — FINAL INVESTMENT SUMMARY
  Latest Date      : 2026-05-08
  Close Price      : 293.05
  RSI              : 72.9
  Sentiment Score  : 0.302
  Signal Score     : -1 / 3
  RECOMMENDATION   : *** HOLD ***
  ─────────────────────────────────────────
  BUY  days :  296 (19.2%)
  HOLD days : 1216 (78.9%)
  SELL days :   30 (1.9%)

  Gold — FINAL INVESTMENT SUMMARY
  Latest Date      : 2026-05-08
  Close Price      : 4720.40
  RSI              : 51.9
  Sentiment Score  : 0.315
  Signal Score     : 1 / 3
  RECOMMENDATION   : *** HOLD ***
  ─────────────────────────────────────────
  BUY  days :  339 (22.0%)
  HOLD days : 1194 (77.3%)
  SELL days :   11 (0.7%)

